In [0]:
# ============================================================

#

# GOLD – dim_employee

# Grain: (source_user_id, project, employee_name)

#

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_TABLE = "workspace.slainte_silver.easyvista_tickets_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_EMPLOYEE = f"{GOLD_DB}.dim_employee"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_silver = spark.table(SILVER_TABLE)

# ============================================================

# 1️⃣ Extract employee base (client + project scoped)

# ============================================================

df_employee_base = (

    df_silver

    .select(

        F.col("source_user_id"),

        F.col("project"),

        F.col("assignee").alias("employee_name")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull() &

        F.col("employee_name").isNotNull()

    )

    .distinct()

)

# ============================================================

# 2️⃣ Generate surrogate key (stable ordering)

# ============================================================

emp_window = Window.orderBy(

    "source_user_id",

    "project",

    "employee_name"

)

df_dim_employee = (

    df_employee_base

    .withColumn("employee_id", F.row_number().over(emp_window))

    .withColumn("employee_active", F.lit(True))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "employee_id",

        "source_user_id",

        "project",

        "employee_name",

        "employee_active"
    )

)

# ============================================================

# 3️⃣ Write GOLD dimension

# ============================================================

df_dim_employee.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_EMPLOYEE)

# ============================================================

# 4️⃣ Preview

# ============================================================

print("✅ dim_employee created successfully")

display(spark.table(DIM_EMPLOYEE).orderBy("source_user_id", "project", "employee_id"))
 